# Secure Data Sharing

Sharing options and base concepts.

## Listing
Offer a share and additional metadata as a data product to one or more accounts on the Snowflake Marketplace.

## Direct Share 
Directly share specific database objects (a share) to another account in the ***same region***.

## Data Exchange
If creating listings that you offer privately to specific accounts isn’t an option.
Set up and manage a group of accounts and offer a share to that group.

## Clean room
- When you use listings, direct shares, and Data Exchange to share data with another party, they can directly access the data.
- Clean room you control how that data is accessed.
- The provider who is sharing their data in a clean room defines what analyses can be run against the shared data.

## Share object

Shares are named Snowflake objects that encapsulate all of the information required to share a database.

You can share the following Snowflake objects:
- Databases
- Tables
- Dynamic tables
- External tables
- Externally managed and managed Apache Iceberg™ tables
- Externally managed Delta Lake tables (with Delta Direct and catalog-linked databases)
- Views:
    - Regular views (only if you force to be secure)
    - Secure views
    - Secure materialized views
    - Semantic views

- Cortex Search services
- User-defined functions (UDFs) (secure and non-secure)
- Models of type USER_MODEL, CORTEX_FINETUNED, or DOC_AI

Important bullets:
 - Virtual warehouse is not part of a share. 
 - If a Snowflake customer consumes a share, they will use their own virtual warehouse.
 - If a non-Snowflake customer is consuming the Share, they will use the data provider compute through a data provider-created virtual warehouse (which would have been separately configured)
 - Consumer cannot use Time Travel on shared data

## Costs
 - no actual data is copied or transferred between accounts. 
 - Shared data does not take up any storage in a consumer account and therefore does not contribute to the consumer’s monthly data storage charges. 
 - The only charges to consumers are for the compute resources (i.e. virtual warehouses) used to query the imported data.


 ## How works

 ![alt](https://docs.snowflake.com/en/_images/data-sharing-overview.png)



 

## Listing vs Direct Share

| Mechanism     | With whom?   | Auto‑Fulfill Across Clouds | Charge for the Data $$ | Offer Data Publicly? | Consumer Usage Metrics |
|---------------|--------------|----------------------------|-------------------------|------------------------|-------------------------|
| Listing       | 1 or many    | Yes                        | Yes                     | Yes                    | Yes                     |
| Direct Share  | 1 or many    | No                         | No                      | No                     | No                      |

## Share important details

### Share data across regions and cloud platforms
- You can share data across regions and cloud platforms, but extra steps are required: https://docs.snowflake.com/en/user-guide/secure-data-sharing-across-regions-platforms

#### Replicate the whole database

1. Set up data replication
    - Create an account in a region where you wish to share data and link it to your local account
    - Enable replication for your accounts
    - Create a replication group and add databases and shares
    - Replicate the group with the databases and shares to the regions where you want to share data with consumers
2. Share data with data consumers
    - Over the new account, apply same sharing actions to the account in the new region / cloud provider.

![alt](https://docs.snowflake.com/en/_images/global-data-sharing-basic.png)

#### Share a subset of data from a database 

To reduce replication costs, they would like to only replicate the relevant rows from their master table.

Steps on source account:
1. Create a database with a subset of data from the database with the source data.
2. Create a secure view with the data to share
3. Create a stream to record changes made to the source table
4. Create a task to insert data into the table in db1 with changes from the source data and start the task.
5. Create a share and grant privileges to the share
6. Create a primary replication group with the database and share

### Multiple databases
- A share can include data from multiple databases: https://docs.snowflake.com/en/user-guide/data-sharing-multiple-db

#### Create and share a secure view in an existing database

A provider who organizes data into different databases based on the characteristics of data and business needs wants to share a secure view in one database that joins data in the database with objects (e.g. schema, table, view) in other databases.

![alt](https://docs.snowflake.com/en/_images/data-sharing-multiple-databases2.png)

#### Create and share a secure view in a separate database

A provider stores customer data in separate databases and does not want to create new objects in those databases. To share data, the provider creates a new database with a secure view. The secure view references objects (schema, table, view) in the databases with customer data.

![alt](https://docs.snowflake.com/en/_images/data-sharing-multiple-databases1.png)
### Changes in the source 
- A share is available immediately to a consumer when you add that consumer’s account to the share.
- New and modified rows are available immediately to consumers who have created a database from the share. This only happens when the consumer already has access.
- A new object created or recreated in a database granted to a share is not automatically available to consumers. You must use the GRANT <privilege> … TO SHARE command to explicitly add the object to the share.

### Secure objects
- For data security and privacy reasons, only secure views are supported in shares at this time. If a standard view is added to a share, Snowflake returns an error
- Creating secure views on streams in your database and then sharing those views with consumers is not recommended. This scenario requires the ability to modify a stream in another account, which is not a supported operation and is therefore an anti-pattern. Instead, allow consumers to create their own streams on the tables and secure views that you share. For more information, see Streams on shared objects (in this topic).
- Do not include secure objects that use the CURRENT_USER or CURRENT_ROLE functions in their definition. The contextual values returned by these functions have no relevance in a consumer’s account and will cause the object to fail when queried/used.

### Storage
Storage lifecycle policies aren’t supported on shared tables.


### Streams on shared objects

Data consumers can create streams in their own databases that record data manipulation language (DML) changes made to the source tables or views.

#### How to enable on producer side:

- Extend the data retention period for the tables
    ALTER TABLE ... DATA_RETENTION_TIME_IN_DAYS 
    Default = 1
    0 - 90 (enterprise or higher)

- Enable change tracking on the shared tables or the underlying tables for a shared view.
    ALTER TABLE ... CHANGE_TRACKING = TRUE

    
### Drop a share

- You can drop (remove) a share at any time. 
- Dropping a share instantly invalidates all databases created from the share by consumer accounts. 
- All queries and other operations performed on these databases no longer work.
- you can recreate it with the same name; however, this does not restore any of the databases created from the share by consumer accounts. 
- The recreated share is treated as a new share and all consumer accounts must create a new database from the new share.

    DROP SHARE


### Non-secure objects bypass

Since secure objects do not allow the same optmization as nonsecure objects, you can choose to bypass this limitation.

To share non-secure views, create a share that allows non-secure objects

     CREATE OR REPLACE SHARE allow_non_secure_views
     SECURE_OBJECTS_ONLY=FALSE
     COMMENT="Share views that require query optimization";


     ALTER SHARE secure_views_only
     SET SECURE_OBJECTS_ONLY = FALSE,
     COMMENT = "Convert to allow sharing non-secure views that require
     query optimization";


    ALTER VIEW secure_view2 UNSET SECURE;


Limitations:

- After you create a share with the SECURE_OBJECTS_ONLY property set to FALSE, you cannot unset this property or set this property to TRUE.
- You can only add non-secure views to shares that have been explicitly configured to allow non-secure objects.


### Enable non-accountadmin to share

- Option 1: Create a database role in a database, grant privileges on objects to the database role, and then grant the database role to the share.

- Option 2: Grant privileges on the database and database objects directly to the share.

- Grant import share:

    - View all INBOUND shares (shared by provider accounts).
    - View all OUTBOUND shares owned by the role.
    - Create databases from inbound shares if the role is also granted the global CREATE DATABASE privilege
    
    USE ROLE ACCOUNTADMIN;
    GRANT IMPORT SHARE ON ACCOUNT TO SYSADMIN;

### Granting roles to the Share

Only database roles can be granted to a share, allowing specific permissions on shared objects within a database to be extended to other accounts.

### Validating Share

The session parameter SIMULATED_DATA_SHARING_CONSUMER only supports secure views and secure materialized views, but does not support secure UDFs. Setting this parameter in a session enables you to simulate querying a secure view as a user in any of the consumer account(s) you plan to share the view with.

    ALTER SESSION SET SIMULATED_DATA_SHARING_CONSUMER = xy12345;


In [ ]:
%%sql -r dataframe_1
Use role sysadmin;
CREATE OR REPLACE DATABASE DATA_S;
use DATA_S.public;
CREATE OR REPLACE STAGE aws_stage
    url='s3://bucketsnowflakes3';

// List files in stage
LIST @aws_stage;


In [ ]:
%%sql -r dataframe_2
// Create table
CREATE OR REPLACE TABLE DATA_S.public.ORDERS (
ORDER_ID	VARCHAR(30)
,AMOUNT	NUMBER(38,0)
,PROFIT	NUMBER(38,0)
,QUANTITY	NUMBER(38,0)
,CATEGORY	VARCHAR(30)
,SUBCATEGORY	VARCHAR(30));

In [ ]:
%%sql -r dataframe_3
// Load data using copy command
COPY INTO DATA_S.public.ORDERS
    FROM @DATA_S.public.aws_stage
    file_format= (type = csv field_delimiter=',' skip_header=1)
    pattern='.*OrderDetails.*';


In [ ]:
%%sql -r dataframe_4
SELECT * FROM DATA_S.public.ORDERS;

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE SECURE VIEW ORDERS_VIEW_SECURE AS
SELECT 
ORDER_ID,
AMOUNT,
QUANTITY
FROM ORDERS
WHERE CATEGORY != 'Furniture'; 

In [ ]:
%%sql -r dataframe_6
-- Create a share object

-- You need the ACCOUNTADMIN role or Create Share 
USE ROLE ACCOUNTADMIN;

-- Create Share
CREATE OR REPLACE SHARE ORDERS_SHARE;


In [ ]:
%%sql -r dataframe_9
--Setup Grants 

// Grant usage on database
GRANT USAGE ON DATABASE DATA_S TO SHARE ORDERS_SHARE; 
// Grant usage on schema
GRANT USAGE ON SCHEMA DATA_S.PUBLIC TO SHARE ORDERS_SHARE; 
// Grant SELECT on table
GRANT SELECT ON TABLE DATA_S.PUBLIC.ORDERS TO SHARE ORDERS_SHARE; 
// Grant select on view
GRANT SELECT ON VIEW  DATA_S.PUBLIC.ORDERS_VIEW_SECURE TO SHARE ORDERS_SHARE;


-- "When sharing data in Snowflake, the Provider (the account sharing the data) must grant the following privileges:
-- SELECT on the specific tables in the database
-- USAGE on the database and schema"


// Validate Grants
SHOW GRANTS TO SHARE ORDERS_SHARE;

### Reader accounts - Direct Share

- Belongs to the provider account that created it. 
- Provider share databases with reader accounts;
- reader account can ***only*** consume data from the provider account that created it.

Refer to the following diagram:

![alt](https://docs.snowflake.com/en/_images/data-sharing-reader.png)

 - Users in a reader account can query data that has been imported with the reader account.
 - but cannot perform any of the DML tasks that are allowed in a full account, such as:
    - data loading
    - insert
    - update

In [ ]:
%%sql -r dataframe_7
-- Create Reader Account --

CREATE MANAGED ACCOUNT reader_account
ADMIN_NAME = read_acc_admin,
ADMIN_PASSWORD = 'Password-123456',
TYPE = READER;


### MANAGED ACCOUNTS

- allows providers to manage and track the reader accounts they have created by using the SHOW MANAGED ACCOUNTS command, ensuring visibility into all reader accounts associated with a provider.
- Reader account has no data of its own.

        SHOW MANAGED ACCOUNTS;

        DROP MANAGED ACCOUNT reader_account

In [ ]:
%%sql -r dataframe_10
--- To drop the account again: DROP MANAGED ACCOUNT reader_account;

// Show accounts
SHOW MANAGED ACCOUNTS;
-- 


In [ ]:
%%sql -r dataframe_11

-- Share the data -- 

ALTER SHARE ORDERS_SHARE 
ADD ACCOUNT = VFB86606;

-- -- Sharing to a lower edition
-- ALTER SHARE ORDERS_SHARE 
-- ADD ACCOUNT =  VFB86606
-- SHARE_RESTRICTIONS=false;

In [ ]:
%%sql -r dataframe_8
//// STEP 4:Create database from share ////
--- By using reader account ---
-- IMPORT SHARE. This privilege allows the user to import shared data from another account, which is necessary when accessing data shared through the Snowflake Marketplace.

// Show all shares (consumer & producers)


SHOW SHARES;

// See details on share
DESC SHARE <consumer_account>.ORDERS_SHARE;

// Create a database in consumer account using the share
CREATE DATABASE DATA_SHARE_DB FROM SHARE <account_producer>.ORDERS_SHARE;

// Validate table access
SELECT * FROM  DATA_SHARE_DB.PUBLIC.ORDERS_VIEW_SECURE;


// Setup virtual warehouse
CREATE WAREHOUSE READ_WH WITH
WAREHOUSE_SIZE='X-SMALL'
AUTO_SUSPEND = 180
AUTO_RESUME = TRUE
INITIALLY_SUSPENDED = TRUE;


## Data Exchange

Provides a data hub for securely collaborating around data with a selected group of members that you invite
![alt](https://docs.snowflake.com/en/_images/private-data-exchange-govern.png)

you can easily provide data to a specific group of consistent business partners taking part in the Data Exchange, such as internal departments in your company or vendors, suppliers, and partners external to your company. If you want to share data with a variety of consumers inside and outside your organization, you can also use listings offered to specific consumers or publicly on the Snowflake Marketplace.

"The necessary privileges for a consumer in the Data Exchange to make a request and receive data are CREATE DATABASE and IMPORT SHARE. These privileges allow the consumer to create a database from the shared data and import the data share into their environment for use.



## Data Sharing Usage
display information about listings published in the Snowflake Marketplace or a data exchange. This includes telemetry data (number of clicks), as well as consumption data (queries run by consumers)

In [ ]:
%%sql -r dataframe_15
select * from snowflake.data_sharing_usage.monetized_usage_daily;

In [ ]:
%%sql -r dataframe_16
select * from snowflake.data_sharing_usage.listing_telemetry_daily;

In [ ]:
%%sql -r dataframe_18
select * from snowflake.data_sharing_usage.APPLICATION_STATE;

## Data Clean Rooms

- Not available in government and VPS deployments.
Data clean rooms are configurable, isolated Snowflake environments where collaborators can import data, specify what queries can be run against that data, and configure data protection settings such as differential privacy and specifying joinable and projectable columns. Access to a clean room is by invitation only.

- Clean rooms don’t support monetization features

https://www.youtube.com/watch?v=FC4Ug95vepM


## Listings

Mechanism that allows you to package, publish, and distribute data, functions, models, or applications through the Snowflake Marketplace or privately to selected consumers.

### Offer Types:
- Public Listing: visible in Marketplace, searchable by everyone.
- Private Listing: you choose exactly which accounts can see/use it.
- Paid Listing: you charge consumers for access.

### Prerequisits using SQL

- Review and accept the Snowflake Provider and Consumer Terms
- Prepare the data for your listing. See Prepare data for a listing.
- Review the Provider Policies.
- Configure account privileges.




In [ ]:
%%sql -r dataframe_22
use role orgadmin;

CREATE EXTERNAL LISTING ORDERS_LISTING
SHARE ORDERS_SHARE AS
$$
 title: "My first SQL listing"
 description: "This is my first listing"
 listing_terms:
   type: "OFFLINE"
 targets:
   accounts: ["GLSNZTB.GAB04792","GLSNZTB.READER_ACCOUNT"]
$$ PUBLISH=FALSE REVIEW=FALSE;

In [ ]:
%%sql -r dataframe_23
ALTER LISTING ORDERS_LISTING PUBLISH;
-- ALTER LISTING ORDERS_LISTING UNPUBLISH;

In [ ]:
%%sql -r dataframe_24
SHOW LISTINGS;

In [ ]:
ALTER LISTING ORDERS_LISTING UNPUBLISH;
DROP LISTING IF EXISTS ORDERS_LISTING;

## Snowflake Marketplace

Where you can explore, access, and provide listings to consumers.

### Data Provider
Use listings on the Snowflake Marketplace to share curated data offerings with many consumers simultaneously, rather than maintain sharing 
relationships with each individual consumer. With Paid listings, you can also charge for your data products.

- Publish listings for free-to-use datasets to generate interest and new opportunities among the Snowflake customer base.
- Publish listings with samples of datasets that can be provided on request or customized for a specific consumer.
- Share live datasets securely and in real-time without creating copies of the data or imposing data integration tasks on the consumer.
- (Preview) Share public listings in Virtual Private Snowflake (VPS) deployments.
- Eliminate the costs of building and maintaining APIs and data pipelines to deliver data to customers.

### Consumer

Use the data provided on the Snowflake Marketplace to explore and access the following:
- Discover and test third-party data sources.
- Receive frictionless access to raw data products from vendors.
- Combine new datasets with your existing data in Snowflake to derive new business insights.
- Have datasets available instantly and updated continually for users.
- Eliminate the costs of building and maintaining various APIs and data pipelines to load and update data.
- Use the business intelligence (BI) tools of your choice.

### Limitations

- The ORGADMIN role is responsible for accepting the Snowflake Consumer Terms of Service, as this role manages organization-wide settings and agreements, including access to the Snowflake Marketplace.


